---

## 📊 Summary: StateGraph Patterns Comparison

| Aspect | Sequential | Conditional | Multi-Turn |
|--------|-----------|-------------|-----------|
| **Use Case** | Fixed pipeline (review → sentiment → reply) | Dynamic routing (question category) | Conversations with memory |
| **Edges** | `add_edge()` only | `add_conditional_edges()` | Both, with `add_messages` |
| **State Flow** | Linear, predictable | Branches based on logic | Accumulates messages |
| **Memory** | Not needed | Not needed | `MemorySaver` + `thread_id` |
| **Example** | Product review pipeline | Question classifier | Travel planning bot |

---

## 🎓 Key Takeaways

### 1. **State Definition is Foundation**
```python
class MyState(TypedDict):
    field1: str
    field2: str
```

### 2. **Nodes Process and Return Updates**
```python
def my_node(state: MyState) -> dict:
    result = process(state)
    return {"field1": result}  # Only return updates
```

### 3. **Edges Connect Nodes**
- **Linear**: `add_edge(from, to)`
- **Conditional**: `add_conditional_edges(from, router_fn, {path: node})`

### 4. **Compile Before Running**
```python
app = workflow.compile(checkpointer=MemorySaver())  # Optional memory
result = app.invoke(initial_state, config={"configurable": {"thread_id": "user_1"}})
```

### 5. **Multi-Turn Requires:**
- `Annotated[list, add_messages]` state type
- `MemorySaver` checkpointer
- Unique `thread_id` per conversation

---

## 🔑 Important Concepts

### START and END
- **START**: Special node representing graph entry
- **END**: Special node representing graph exit
- Every workflow must connect START to END

### Literal Type Hints
- Used for conditional routing return values
- Ensures type-safe branching
- Example: `Literal["node1", "node2", "node3"]`

### Message Reducer
- `add_messages`: Appends new messages to history
- Automatically handles list concatenation
- Essential for multi-turn conversations

### Temperature Settings
- **temperature=0.0**: Deterministic (for classification)
- **temperature=0.7**: Creative (for generation)
- Use `llm.with_config(configurable={"temperature": value})`

---

## 🚀 Best Practices

1. **Use TypedDict for clear state structure**
   - Self-documenting
   - Type-safe
   - IDE autocomplete

2. **Keep nodes focused**
   - One responsibility per node
   - Easier to test and debug

3. **Use MemorySaver for production conversations**
   - Enables multi-turn state persistence
   - Switch to Redis/DB for scalability

4. **Set temperature appropriately**
   - Low (0.0) for deterministic decisions
   - Higher (0.7-1.0) for creative outputs

5. **Test with small inputs first**
   - Verify node logic before full pipeline
   - Debug routing before complex workflows

6. **Use meaningful node names**
   - Reflects functionality
   - Aids in debugging

---

## 📚 Related Files in Your Workspace

Based on your assignments, explore these for deeper learning:

- [GEN-AI-Prog-HandsOn/Agentic AI/Day3/task02.ipynb](../Day3/task02.ipynb) - Sequential pipeline
- [GEN-AI-Prog-HandsOn/Agentic AI/Day3/task03.ipynb](../Day3/task03.ipynb) - Conditional routing
- [GEN-AI-Prog-HandsOn/Agentic AI/Day4/travel_planner.ipynb](../Day4/travel_planner.ipynb) - Multi-turn agent
- [GEN-AI-Prog-HandsOn/Agentic AI/Day4/expense_calculator.ipynb](../Day4/expense_calculator.ipynb) - Practical application
- [GEN-AI-Prog-HandsOn/Agentic AI/Day5/customer_support.ipynb](../Day5/customer_support.ipynb) - Advanced patterns

---

## 🎯 Next Steps

1. **Modify the examples**: Change prompts, states, and logic
2. **Combine patterns**: Mix sequential and conditional edges
3. **Add error handling**: Gracefully handle LLM failures
4. **Implement tools**: Extend nodes to call external APIs
5. **Deploy**: Move from MemorySaver to production storage

---

## 📝 Conclusion

StateGraph provides a powerful abstraction for building agentic workflows:
- ✅ **Clear state management** through TypedDict
- ✅ **Flexible routing** with conditional edges
- ✅ **Multi-turn support** with message history
- ✅ **Production-ready** with checkpointing

By mastering these patterns, you can build sophisticated AI systems that reason, decide, and interact intelligently!

---

**Created by: Nandavardhan Doodala**  
**Topic: StateGraph Implementation Walkthrough**  
**Date: August 2026**

In [ ]:
# ==========================================
# EXAMPLE: Multi-Turn Travel Planning Agent
# ==========================================

# Define multi-turn conversation state
class TravelState(TypedDict):
    """State for multi-turn travel planning conversation"""
    messages: Annotated[list[BaseMessage], add_messages]  # Auto-append messages
    preferences: dict  # Track user preferences across turns


# Define the travel planner agent node
def travel_planner_agent(state: TravelState):
    """Multi-turn travel planning assistant"""
    system_prompt = SystemMessage(
        content=(
            "You are a helpful travel assistant. Your job is to:\n"
            "1. Remember user preferences (destination, dates, budget)\n"
            "2. Ask clarifying questions when needed\n"
            "3. Provide tailored travel recommendations\n"
            "Always be conversational and helpful."
        )
    )
    
    # Combine system prompt with message history
    messages = [system_prompt] + state["messages"]
    
    # Get response from LLM
    response = llm.invoke(messages)
    
    # Return new message appended to history
    return {"messages": [response]}


# Build multi-turn workflow
workflow_multiturn = StateGraph(TravelState)
workflow_multiturn.add_node("travel_agent", travel_planner_agent)
workflow_multiturn.add_edge(START, "travel_agent")
workflow_multiturn.add_edge("travel_agent", END)

# Compile with MemorySaver for multi-turn state persistence
checkpointer = MemorySaver()
app_multiturn = workflow_multiturn.compile(checkpointer=checkpointer)

print("✅ Multi-turn travel agent compiled with MemorySaver")
print("\n" + "="*60)
print("RUNNING MULTI-TURN CONVERSATION: Travel Planning")
print("="*60)

# Conversation thread ID (unique per conversation)
thread_id = "user_123_travel_plan"

# Turn 1: User provides initial request
print("\n--- TURN 1: User provides initial request ---")
turn1_input = {
    "messages": [HumanMessage(content="I want to plan a trip to Europe. Can you help?")],
    "preferences": {}
}
result_turn1 = app_multiturn.invoke(turn1_input, config={"configurable": {"thread_id": thread_id}})
print(f"User: I want to plan a trip to Europe. Can you help?")
print(f"Assistant: {result_turn1['messages'][-1].content}")

# Turn 2: User provides more details
print("\n--- TURN 2: User provides budget and dates ---")
turn2_input = {
    "messages": [HumanMessage(content="I have 2 weeks and a budget of $3000")],
    "preferences": {}
}
result_turn2 = app_multiturn.invoke(turn2_input, config={"configurable": {"thread_id": thread_id}})
print(f"User: I have 2 weeks and a budget of $3000")
print(f"Assistant: {result_turn2['messages'][-1].content}")

# Turn 3: User asks for specific destination
print("\n--- TURN 3: User asks for recommendation ---")
turn3_input = {
    "messages": [HumanMessage(content="What countries would you recommend?")],
    "preferences": {}
}
result_turn3 = app_multiturn.invoke(turn3_input, config={"configurable": {"thread_id": thread_id}})
print(f"User: What countries would you recommend?")
print(f"Assistant: {result_turn3['messages'][-1].content}")

print("\n" + "="*60)
print("✅ Multi-turn conversation completed!")
print(f"   Total messages in conversation: {len(result_turn3['messages'])}")
print("="*60)

---

## Section 7️⃣: Execute Multi-turn Interactions

For conversational agents, you need to:
1. **Persist state** across multiple turns using `MemorySaver` checkpointer
2. **Use `thread_id`** to identify unique conversations
3. **Accumulate messages** using `add_messages` reducer

### Key Concepts:
- **thread_id**: Unique identifier for a conversation session
- **MemorySaver**: In-memory state checkpointer (use Redis/DB in production)
- **add_messages**: Automatically appends new messages to history

---

### Example: Multi-Turn Travel Planning Agent

In [ ]:
# ==========================================
# EXAMPLE 2: Compile and Run Conditional Router
# ==========================================

# Compile the router workflow
app_router = workflow_router.compile()

print("✅ Router graph compiled and ready to run")
print("\n" + "="*60)
print("RUNNING CONDITIONAL ROUTER: Multi-Category Question Answerer")
print("="*60)

# Test with different question categories
test_questions = [
    "Why is the sky blue and how does light scattering work?",     # Science
    "Who was Julius Caesar and what was his impact on Rome?",      # History
    "What's the best way to learn Python programming?"             # General
]

for i, question in enumerate(test_questions, 1):
    print(f"\n--- Test Case {i} ---")
    print(f"Question: {question}")
    
    result_router = app_router.invoke({"question": question})
    
    print(f"Routed to: {result_router['category']} persona")
    print(f"Response: {result_router['response'][:150]}...")  # Show first 150 chars

print("\n" + "="*60)

### Example 2: Compile and Run Conditional Router

In [ ]:
# ==========================================
# EXAMPLE 1: Compile and Run Sequential Pipeline
# ==========================================

# Compile the sequential workflow
app_sequential = workflow_sequential.compile()

print("✅ Sequential graph compiled and ready to run")
print("\n" + "="*60)
print("RUNNING SEQUENTIAL PIPELINE: Product Review Generator")
print("="*60)

# Test input
initial_state = {"product": "wireless noise-cancelling headphones"}

# Invoke the graph
result_sequential = app_sequential.invoke(initial_state)

# Display results
print(f"\n📦 Product: {result_sequential['product']}")
print(f"\n📝 Generated Review:\n{result_sequential['review']}")
print(f"\n😊 Detected Sentiment: {result_sequential['sentiment']}")
print(f"\n💬 Official Brand Reply:\n{result_sequential['reply']}")
print("\n" + "="*60)

---

## Section 6️⃣: Compile and Test the Graph

Before running, the graph must be **compiled**. Compilation transforms the graph definition into an executable object.

### Optional: Add Memory Checkpointer
- **MemorySaver**: Stores state snapshots in memory for multi-turn conversations
- Enables state retrieval using `thread_id`

### Key Methods:
- `workflow.compile()` - Basic compilation
- `workflow.compile(checkpointer=MemorySaver())` - With memory/state persistence

---

### Example 1: Compile and Run Sequential Pipeline

In [ ]:
# ==========================================
# EXAMPLE 2: Add Conditional Edges to Router Graph
# ==========================================

# Router function decides which node executes based on question category
workflow_router.add_conditional_edges(
    START,                          # Start from START
    route_question,                 # Use router function to decide
    {
        "science_node": "science_node",
        "history_node": "history_node",
        "general_node": "general_node"
    }
)

# All branches converge to END
workflow_router.add_edge("science_node", END)
workflow_router.add_edge("history_node", END)
workflow_router.add_edge("general_node", END)

print("✅ Conditional edges added")
print("   START → route_question() → [science_node | history_node | general_node] → END")

### Example 2: Add Conditional Edges to Router Graph

In [ ]:
# ==========================================
# EXAMPLE 1: Add Edges to Sequential Graph
# ==========================================

# Flow: START → review_generator → sentiment_classifier → reply_generator → END

workflow_sequential.add_edge(START, "review_generator")
workflow_sequential.add_edge("review_generator", "sentiment_classifier")
workflow_sequential.add_edge("sentiment_classifier", "reply_generator")
workflow_sequential.add_edge("reply_generator", END)

print("✅ Sequential edges added")
print("   START → review_generator → sentiment_classifier → reply_generator → END")

---

## Section 5️⃣: Add Edges and Conditional Logic

**Edges** connect nodes together. There are two types:

### 1. **Linear Edges** (`add_edge()`)
- Fixed connection: always goes from one node to another
- Used in sequential pipelines

### 2. **Conditional Edges** (`add_conditional_edges()`)
- Dynamic routing based on state
- Router function decides which node executes next
- Returns a `Literal` type

### Special Nodes:
- **START**: Entry point of the graph
- **END**: Exit point of the graph

---

### Example 1: Add Edges to Sequential Graph

In [ ]:
# ==========================================
# EXAMPLE 2: Build Conditional Router Graph
# ==========================================

# Initialize StateGraph with RouterState
workflow_router = StateGraph(RouterState)

# Register nodes
workflow_router.add_node("science_node", science_node)
workflow_router.add_node("history_node", history_node)
workflow_router.add_node("general_node", general_node)

print("✅ Router graph nodes added")
print("   - science_node")
print("   - history_node")
print("   - general_node")

### Example 2: Build Conditional Router Graph

In [ ]:
# ==========================================
# EXAMPLE 1: Build Sequential Graph
# ==========================================

# Initialize StateGraph with ReviewState
workflow_sequential = StateGraph(ReviewState)

# Register nodes
workflow_sequential.add_node("review_generator", review_node)
workflow_sequential.add_node("sentiment_classifier", sentiment_node)
workflow_sequential.add_node("reply_generator", reply_node)

print("✅ Sequential graph nodes added")
print("   - review_generator")
print("   - sentiment_classifier")
print("   - reply_generator")

---

## Section 4️⃣: Build the StateGraph

The StateGraph is initialized with a **State class** and then nodes are added to it using `add_node()`.

### Basic Structure:
```python
workflow = StateGraph(StateClass)
workflow.add_node("node_name", node_function)
```

---

### Example 1: Build Sequential Graph

In [ ]:
# ==========================================
# EXAMPLE 3: Router Function
# ==========================================

def route_question(state: RouterState) -> Literal["science_node", "history_node", "general_node"]:
    """
    Classify question and route to appropriate node.
    Must return a Literal type matching node names.
    """
    prompt = (
        f"Classify this question as: science, history, or general\n"
        f"Question: {state['question']}\n"
        f"Respond with ONLY the category word."
    )
    # Temperature 0 for deterministic routing
    classifier = llm.with_config(configurable={"temperature": 0.0})
    category = classifier.invoke(prompt).content.strip().lower()
    
    if "science" in category:
        return "science_node"
    elif "history" in category:
        return "history_node"
    else:
        return "general_node"

print("✅ Router function defined")

### Example 3: Router Function

The **router** function classifies state and returns which node to execute next. It must return a `Literal` type.

In [ ]:
# ==========================================
# EXAMPLE 2: Conditional Routing Nodes
# ==========================================

def science_node(state: RouterState) -> dict:
    """Respond as an enthusiastic science teacher"""
    prompt = f"You are a science teacher. Answer: {state['question']}"
    response = llm.invoke(prompt)
    return {"response": response.content, "category": "SCIENCE"}


def history_node(state: RouterState) -> dict:
    """Respond as a knowledgeable historian"""
    prompt = f"You are a historian. Answer: {state['question']}"
    response = llm.invoke(prompt)
    return {"response": response.content, "category": "HISTORY"}


def general_node(state: RouterState) -> dict:
    """Respond as a general assistant"""
    prompt = f"You are a helpful assistant. Answer: {state['question']}"
    response = llm.invoke(prompt)
    return {"response": response.content, "category": "GENERAL"}

print("✅ Conditional routing nodes defined (science, history, general)")

### Example 2: Conditional Routing Nodes

Nodes that handle different topics with persona-based responses.

In [ ]:
# ==========================================
# EXAMPLE 1: Sequential Pipeline Nodes
# ==========================================

def review_node(state: ReviewState) -> dict:
    """Generate a product review"""
    prompt = f"Write a brief, realistic 2-3 sentence review for: {state['product']}"
    response = llm.invoke(prompt)
    return {"review": response.content}


def sentiment_node(state: ReviewState) -> dict:
    """Classify review sentiment as Positive, Negative, or Neutral"""
    prompt = (
        f"Analyze sentiment:\n'{state['review']}'\n\n"
        "Respond with ONLY one word: Positive, Negative, or Neutral"
    )
    # Use temperature=0 for deterministic classification
    classifier = llm.with_config(configurable={"temperature": 0.0})
    response = classifier.invoke(prompt)
    sentiment = response.content.strip().replace(".", "")
    return {"sentiment": sentiment}


def reply_node(state: ReviewState) -> dict:
    """Generate brand response based on sentiment"""
    prompt = (
        f"You are a customer service representative.\n"
        f"Sentiment: {state['sentiment']}\n"
        f"Review: {state['review']}\n\n"
        f"Write a professional one-line response."
    )
    response = llm.invoke(prompt)
    return {"reply": response.content}

print("✅ Sequential pipeline nodes defined (review → sentiment → reply)")

---

## Section 3️⃣: Create Node Functions

**Nodes** are functions that:
1. Receive the current state
2. Process it (call LLM, compute logic, etc.)
3. Return a dictionary of updated fields

### Rule: Return a dict, not the full state
```python
def my_node(state: MyState) -> dict:
    # Process state
    result = "processed"
    return {"field": result}  # ← Return only updates
```

---

### Example 1: Sequential Pipeline Nodes

Nodes for product review pipeline: review → sentiment → reply

In [ ]:
# Example 3: Multi-Turn Conversation State
class ConversationState(TypedDict):
    """State for multi-turn conversations with message history"""
    # add_messages: automatically appends new messages instead of replacing
    messages: Annotated[list[BaseMessage], add_messages]
    user_preferences: dict  # Tracks user details across turns

print("✅ ConversationState defined")
print("   Fields: messages (with auto-append), user_preferences")

### Example 3: Multi-Turn Conversation State

This state persists across multiple turns and uses `add_messages` for automatic history appending.

In [ ]:
# Example 2: State for Conditional Routing
class RouterState(TypedDict):
    """State for question classification and routing"""
    question: str       # Input question
    category: str       # Classified category (science, history, general)
    response: str       # Generated response

print("✅ RouterState defined")
print("   Fields: question, category, response")

### Example 2: Conditional Routing State

This state is used for routing questions to different personas (science, history, general).

In [ ]:
# Example 1: Simple State for Sequential Pipeline
class ReviewState(TypedDict):
    """State for product review pipeline"""
    product: str        # Input: product name
    review: str         # Generated review
    sentiment: str      # Classified sentiment
    reply: str          # Brand response

print("✅ ReviewState defined")
print("   Fields: product, review, sentiment, reply")

---

## Section 2️⃣: Define State Schema

The **State Schema** defines what data flows through your graph. Think of it as a container that holds information as it passes from node to node.

### Key Concepts:
- **TypedDict**: A Python type hint for dictionaries with specific keys and value types
- **State Fields**: Each field gets added/updated as nodes process it
- **Immutable**: The state object is immutable; nodes return new fields to update it

### Example 1: Simple Sequential State

This state is used for a review pipeline (product → review → sentiment → reply).

In [ ]:
# ==========================================
# Load Environment Variables & Initialize LLM
# ==========================================

load_dotenv()

# Extract API credentials
api_key = os.getenv("KEY")
base_url = os.getenv("BASE_URL")
model_name = os.getenv("MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

if not api_key:
    raise ValueError("❌ 'KEY' not found in your .env file")

# Initialize the LLM client
llm = ChatAnthropic(
    model=model_name,
    anthropic_api_key=api_key,
    anthropic_api_url=base_url,
    temperature=0.7
)

print(f"✅ LLM Initialized!")
print(f"   Model: {model_name}")
print(f"   Base URL: {base_url}")

### API Configuration

Before running the code, create a `.env` file in your project directory with the following content:

```
KEY=sk-zK7xMXa2pANc64xuf4oaTA
BASE_URL=https://llmgw-wp.tekstac.com
MODEL=global.anthropic.claude-haiku-4-5-20251001-v1:0
```

Available models:
- `global.anthropic.claude-haiku-4-5-20251001-v1:0` (Fast, cost-effective)
- `global.anthropic.claude-opus-4-5-20251101-v1:0` (Balanced)
- `global.anthropic.claude-sonnet-4-6` (Latest, high quality)

In [ ]:
# ==========================================
# Import Required Libraries
# ==========================================

from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
import os

# LangGraph core components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# LangChain components
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage

print("✅ All imports successful!")

In [ ]:
# Install required packages
import subprocess
import sys

packages = ["langgraph", "langchain-anthropic", "python-dotenv", "anthropic"]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✅ {package} is already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

print("\n✅ All required packages are installed!")

## Section 1️⃣: Import Required Libraries

Install and import all necessary packages for StateGraph workflows.

# StateGraph Implementation Walkthrough 🤖

## Presenter: Nandavardhan Doodala
## Topic: StateGraph Implementation Walkthrough

### 📚 Overview
This comprehensive guide demonstrates how to build stateful AI workflows using LangGraph's `StateGraph`. You'll learn:
- ✅ Core StateGraph concepts and architecture
- ✅ Defining state schemas with TypedDict
- ✅ Creating and connecting node functions
- ✅ Sequential and conditional edge patterns
- ✅ Managing multi-turn conversations with memory
- ✅ Real-world applications and best practices

### 🎯 Learning Outcomes
By the end of this walkthrough, you will understand:
1. How StateGraph structures AI workflows
2. How to define state and pass data between nodes
3. How to implement sequential pipelines
4. How to create conditional routing logic
5. How to persist state across multiple turns
6. How to build production-ready agentic systems

---